<a href="https://colab.research.google.com/github/jakkrol/machine-learning/blob/main/DeepfakeDetector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os

os.environ['KAGGLE_USERNAME'] = ""
os.environ['KAGGLE_KEY'] = ""


!mkdir -p ~/.kaggle
import json
with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
    json.dump({'username': os.environ['KAGGLE_USERNAME'], 'key': os.environ['KAGGLE_KEY']}, f)

!chmod 600 ~/.kaggle/kaggle.json
print("Kaggle zostało skonfigurowane bezpośrednio z kodu!")

Kaggle zostało skonfigurowane bezpośrednio z kodu!


In [2]:
# 1. Pobieramy zidentyfikowany dataset (używamy dokładnej nazwy z tabeli)
!kaggle datasets download -d kshitizbhargava/deepfake-face-images

# 2. Rozpakowujemy pliki
import zipfile
import os

print("Rozpakowywanie paczki deepfake-face-images.zip...")
with zipfile.ZipFile("deepfake-face-images.zip", 'r') as zip_ref:
    zip_ref.extractall("deepfake_faces_data")

print("Rozpakowywanie zakończone pomyślnie!")

# 3. Sprawdzamy strukturę folderów wewnątrz 'deepfake_faces_data'
base_dir = 'deepfake_faces_data'
if os.path.exists(base_dir):
    print("\nStruktura katalogów w środku:")
    print(os.listdir(base_dir))

Dataset URL: https://www.kaggle.com/datasets/kshitizbhargava/deepfake-face-images
License(s): MIT
100% 244M/244M [00:01<00:00, 143MB/s]

Rozpakowywanie paczki deepfake-face-images.zip...
Rozpakowywanie zakończone pomyślnie!

Struktura katalogów w środku:
['Final Dataset']


In [3]:
import os

# Definiujemy ścieżkę do rozpakowanego folderu
dataset_path = 'deepfake_faces_data/Final Dataset'

if os.path.exists(dataset_path):
    print("Zawartość folderu 'Final Dataset':")
    subfolders = os.listdir(dataset_path)
    print(subfolders)

    # Sprawdźmy głębiej, co jest w środku pierwszego z brzegu podfolderu
    for folder in subfolders:
        full_sub_path = os.path.join(dataset_path, folder)
        if os.path.isdir(full_sub_path):
            print(f"\nLiczba plików w folderze '{folder}': {len(os.listdir(full_sub_path))}")
            # Pokazuje 3 przykładowe nazwy plików
            print(f"Przykładowe pliki: {os.listdir(full_sub_path)[:3]}")
else:
    print("Coś poszło nie tak, nie znaleziono folderu 'Final Dataset'.")

Zawartość folderu 'Final Dataset':
['Real', 'Fake', 'dataset.csv']

Liczba plików w folderze 'Real': 5890
Przykładowe pliki: ['02683.jpg', 'real_43_aug_3.jpg', '03367.jpg']

Liczba plików w folderze 'Fake': 7000
Przykładowe pliki: ['fake_165_aug_3.jpg', 'fake_592_aug_4.jpg', 'fake_311_aug_0.jpg']


In [4]:
import tensorflow as tf
from tensorflow.keras.utils import image_dataset_from_directory

base_dir = 'deepfake_faces_data/Final Dataset'
BATCH_SIZE = 32
IMG_SIZE = (128, 128)

print("\n--- ładowanie zbioru treningowego ---")
train_ds = image_dataset_from_directory(
    base_dir,
    validation_split = 0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

print("\n--- ładowanie zbioru testowego ---")
val_ds = image_dataset_from_directory(
    base_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
print(f"\nWykryte klasy: {class_names} (Gdzie: {class_names[0]} = Prawdziwe, {class_names[1]} = Deepfake)")


--- ładowanie zbioru treningowego ---
Found 12890 files belonging to 2 classes.
Using 10312 files for training.

--- ładowanie zbioru testowego ---
Found 12890 files belonging to 2 classes.
Using 2578 files for validation.

Wykryte klasy: ['Fake', 'Real'] (Gdzie: Fake = Prawdziwe, Real = Deepfake)


In [5]:
from tensorflow.keras import layers, models

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

base_model = tf.keras.applications.MobileNetV2(
    input_shape = (128,128,3),
    include_top = False,
    weights = "imagenet"
)

base_model.trainable = False

model = models.Sequential([
    # Warstwa przetwarzania wstępnego (skaluje piksele z zakresu 0-255 do wymaganego przez model -1 do 1)
    layers.Lambda(tf.keras.applications.mobilenet_v2.preprocess_input, input_shape=(128, 128, 3)),

    # Nasz zamrożony mózg bazowy
    base_model,

    # Spłaszczamy mapy cech z grafiki do jednego wektora
    layers.GlobalAveragePooling2D(),

    # Warstwa zabezpieczająca przed przeuczeniem (odrzuca losowo 20% połączeń)
    layers.Dropout(0.2),

    # WARSTWA WYJŚCIOWA: 1 neuron, funkcja 'sigmoid' (zwraca wynik od 0.0 do 1.0)
    # Blisko 0 = Prawdziwe zdjęcie, Blisko 1 = Deepfake
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',                                     # Najpopularniejszy, stabilny algorytm uczący
    loss='binary_crossentropy',                           # Standardowa funkcja strat dla problemów Tak/Nie
    metrics=['accuracy']                                  # Chcemy na bieżąco widzieć skuteczność w %
)

model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/lambda_layer.py:65: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lambda (Lambda)                 │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_128            │ (None, 4, 4, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,259,265 (8.62 MB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [6]:
print("--start training--")

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

--start training--
Epoch 1/10
323/323 ━━━━━━━━━━━━━━━━━━━━ 55s 112ms/step - accuracy: 0.7322 - loss: 0.5310 - val_accuracy: 0.7622 - val_loss: 0.4697
Epoch 2/10
323/323 ━━━━━━━━━━━━━━━━━━━━ 9s 29ms/step - accuracy: 0.7852 - loss: 0.4509 - val_accuracy: 0.7851 - val_loss: 0.4297
Epoch 3/10
323/323 ━━━━━━━━━━━━━━━━━━━━ 11s 34ms/step - accuracy: 0.7981 - loss: 0.4233 - val_accuracy: 0.7859 - val_loss: 0.4243
Epoch 4/10
323/323 ━━━━━━━━━━━━━━━━━━━━ 21s 35ms/step - accuracy: 0.8062 - loss: 0.4070 - val_accuracy: 0.8061 - val_loss: 0.4011
Epoch 5/10
323/323 ━━━━━━━━━━━━━━━━━━━━ 11s 33ms/step - accuracy: 0.8140 - loss: 0.4022 - val_accuracy: 0.8064 - val_loss: 0.4001
Epoch 6/10
323/323 ━━━━━━━━━━━━━━━━━━━━ 9s 29ms/step - accuracy: 0.8144 - loss: 0.3983 - val_accuracy: 0.7839 - val_loss: 0.4261
Epoch 7/10
323/323 ━━━━━━━━━━━━━━━━━━━━ 10s 28ms/step - accuracy: 0.8180 - loss: 0.3969 - val_accuracy: 0.8084 - val_loss: 0.3938
Epoch 8/10
323/323 ━━━━━━━━━━━━━━━━━━━━ 9s 27ms/step - accuracy: 0.8144 

In [8]:
model.save('deepfake_v1.keras')

from google.colab import files
files.download('deepfake_v1.keras')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>